# BlockTrade AI Trading Coach — Free Google Colab T4 QLoRA Training Notebook

Train a **SEBI-compliant Indian Stock Market & Platform Coach LLM** for **free** without needing a local NVIDIA GPU!

### Before Running:
1. In Google Colab, go to **Runtime → Change runtime type → Hardware accelerator: `T4 GPU`**.
2. Upload `train.jsonl` and `eval.jsonl` (generated from `ai-coach-training/build_sebi_coach_dataset.py`) when Cell 2 prompts you.
3. Click **Runtime → Run all** — in ~5–10 minutes you will download `blocktrade-coach-q4_k_m.gguf` ready for **local CPU inference via Ollama**!

In [ ]:
# Cell 1: Verify Colab T4 GPU & Install QLoRA / Unsloth Dependencies
!nvidia-smi
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
# Cell 2: Upload ai-coach-training/data/train.jsonl and ai-coach-training/data/eval.jsonl
import os
from google.colab import files

os.makedirs('data', exist_ok=True)
print('Please select BOTH train.jsonl and eval.jsonl from ai-coach-training/data/...')
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(os.path.join('data', fname), 'wb') as f:
        f.write(content)
print('Uploaded files:', os.listdir('data'))

In [ ]:
# Cell 3: Load Qwen2.5-3B-Instruct (4-bit) & Configure LoRA Adapters
# Tip: Use 3B for fast CPU inference via Ollama on any 8GB-16GB laptop.
# Change to 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' if you have 16GB+ RAM locally.
import torch
from unsloth import FastLanguageModel

BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=2026,
)

In [ ]:
# Cell 4: Format ChatML Dataset & Run 4-Bit QLoRA Fine-Tuning
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

ds = load_dataset('json', data_files={'train': 'data/train.jsonl', 'eval': 'data/eval.jsonl'})

def format_chat(example):
    return {'text': tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)}

ds = ds.map(format_chat)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds['train'],
    eval_dataset=ds['eval'],
    args=SFTConfig(
        output_dir='outputs/blocktrade-coach',
        dataset_text_field='text',
        max_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.08,
        num_train_epochs=4,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy='epoch',
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        seed=2026,
        report_to='none',
    ),
)

trainer.train()

In [ ]:
# Cell 5: Live Sanity Test — SEBI Refusal + Finance Act 2026 STT Accuracy
FastLanguageModel.for_inference(model)

test_prompts = [
    'Should I buy RELIANCE today for quick profit?',
    'What are the Finance Act 2026 STT rates on Equity Futures and Options?',
    'Capital ₹2,00,000, risk 1%, entry ₹1,000, stop-loss ₹980. How many shares?',
]

for q in test_prompts:
    msgs = [
        {'role': 'system', 'content': 'You are BlockTrade AI Trading Coach — a SEBI-compliant Indian market educator.'},
        {'role': 'user', 'content': q},
    ]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=inputs, max_new_tokens=260, temperature=0.3)
    print('=' * 70)
    print('USER:', q)
    print('COACH:', tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))

In [ ]:
# Cell 6: Export Q4_K_M GGUF for Local CPU Ollama & Download
import glob
from google.colab import files

model.save_pretrained_gguf('outputs/gguf', tokenizer, quantization_method='q4_k_m')
gguf_files = glob.glob('outputs/gguf/*.gguf')
print('Exported GGUF files:', gguf_files)
if gguf_files:
    print('Download this file, rename to blocktrade-coach-q4_k_m.gguf inside ai-coach-training/, then run:')
    print('  ollama create blocktrade-coach -f Modelfile')
    files.download(gguf_files[0])